In [1]:
#[https://polymarket.com/event/strait-of-hormuz-traffic-returns-to-normal-by-october-31-20260810151043583](https://polymarket.com/event/strait-of-hormuz-traffic-returns-to-normal-by-october-31-20260810151043583)

SyntaxError: invalid syntax (1526758181.py, line 1)

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
from polytrader.data import load_history, fetch_price_history
import pandas as pd

In [5]:
DAYS = 90
BUCKET_SECONDS = 300

markets = {
    "oct_31": "strait-of-hormuz-traffic-returns-to-normal-by-october-31-20260810151043583",
    "nov_30": "strait-of-hormuz-traffic-returns-to-normal-by-november-30-20260810151158765",
    "dec_31": "strait-of-hormuz-traffic-returns-to-normal-by-december-31",
}

histories = {}
series = {}

for name, slug in markets.items():
    history = fetch_price_history(
        event=slug,
        market=slug,
        days=DAYS,
        bucket_seconds=BUCKET_SECONDS,
    )
    histories[name] = history

    df = pd.DataFrame(history.data)

    # Support either timestamp/price or Polymarket's t/p field names.
    df = df.rename(columns={"t": "timestamp", "p": "price"})

    if df.empty:
        raise ValueError(f"No history returned for {name}")

    if not {"timestamp", "price"}.issubset(df.columns):
        raise ValueError(f"Unexpected columns for {name}: {df.columns.tolist()}")

    # Handle Unix timestamps in seconds or datetime/ISO strings.
    numeric_time = pd.to_numeric(df["timestamp"], errors="coerce")
    if numeric_time.notna().all():
        df["timestamp"] = pd.to_datetime(numeric_time, unit="s", utc=True)
    else:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    df["price"] = pd.to_numeric(df["price"], errors="raise")

    s = (
        df.sort_values("timestamp")
        .drop_duplicates("timestamp", keep="last")
        .set_index("timestamp")["price"]
    )

    # Last observation in each 5-minute interval.
    # Label intervals by their END so observations precede the row timestamp.
    series[name] = s.resample(
        f"{BUCKET_SECONDS}s",
        closed="right",
        label="right",
    ).last()

    print(f"{name}: {len(df):,} observations")

prices = pd.concat(series, axis=1).sort_index()
prices.index.name = "timestamp"

# Exclude the current, unfinished interval.
cutoff = pd.Timestamp.now(tz="UTC").floor(f"{BUCKET_SECONDS}s")
prices = prices.loc[:cutoff]

# Keep missing observations visible; don't invent prices by filling gaps.
display(prices.tail(20).style.format("{:.2%}", na_rep="—"))

# Check how much overlapping data you have.
display(pd.DataFrame({
    "observations": prices.count(),
    "missing_buckets": prices.isna().sum(),
}))

aligned = prices.dropna().copy()
print(f"Complete overlapping rows: {len(aligned):,}")

oct_31: 13,058 observations
nov_30: 13,057 observations
dec_31: 14,919 observations


C:\Users\bekae\AppData\Local\Temp\ipykernel_27992\3604168015.py:58: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  prices = pd.concat(series, axis=1).sort_index()


,oct_31,nov_30,dec_31
timestamp,,,
2026-09-25 20:35:00+00:00,10.50%,17.50%,27.50%
2026-09-25 20:40:00+00:00,10.50%,17.50%,27.50%
2026-09-25 20:45:00+00:00,10.50%,17.50%,27.50%
2026-09-25 20:50:00+00:00,10.50%,17.50%,27.50%
2026-09-25 20:55:00+00:00,10.50%,17.50%,27.50%
2026-09-25 21:00:00+00:00,10.50%,17.50%,27.50%
2026-09-25 21:05:00+00:00,10.50%,17.50%,27.50%
2026-09-25 21:10:00+00:00,10.50%,17.50%,27.50%
2026-09-25 21:15:00+00:00,10.50%,17.50%,27.50%


,observations,missing_buckets
oct_31,13058,5641
nov_30,13056,5643
dec_31,14918,3781


Complete overlapping rows: 13,055


In [8]:
analysis = aligned.copy()

# Deadline spreads, expressed in percentage points.
analysis["nov_minus_oct_pp"] = (
    analysis["nov_30"] - analysis["oct_31"]
) * 100

analysis["dec_minus_nov_pp"] = (
    analysis["dec_31"] - analysis["nov_30"]
) * 100

# Implied conditional probability of occurrence between deadlines.
analysis["nov_given_not_oct"] = (
    (analysis["nov_30"] - analysis["oct_31"])
    / (1 - analysis["oct_31"]).replace(0, float("nan"))
)

analysis["dec_given_not_nov"] = (
    (analysis["dec_31"] - analysis["nov_30"])
    / (1 - analysis["nov_30"]).replace(0, float("nan"))
)

# Flag historical prices that violate the expected deadline ordering.
analysis["ordering_violation"] = (
    (analysis["oct_31"] > analysis["nov_30"])
    | (analysis["nov_30"] > analysis["dec_31"])
)

formats = {
    col: "{:.2%}"
    for col in [
        "oct_31", "nov_30", "dec_31",
        "nov_given_not_oct", "dec_given_not_nov",
    ]
}
formats.update({
    "nov_minus_oct_pp": "{:.2f}",
    "dec_minus_nov_pp": "{:.2f}",
})

display(analysis[analysis["ordering_violation"]].tail(20).style.format(formats, na_rep="—"))

,oct_31,nov_30,dec_31,nov_minus_oct_pp,dec_minus_nov_pp,nov_given_not_oct,dec_given_not_nov,ordering_violation
timestamp,,,,,,,,
2026-08-11 09:20:00+00:00,26.00%,50.00%,46.50%,24.00,-3.50,32.43%,-7.00%,True
2026-08-11 09:25:00+00:00,26.00%,50.00%,46.50%,24.00,-3.50,32.43%,-7.00%,True
2026-08-11 09:30:00+00:00,33.00%,52.00%,46.50%,19.00,-5.50,28.36%,-11.46%,True
2026-08-11 09:35:00+00:00,33.50%,52.00%,46.50%,18.50,-5.50,27.82%,-11.46%,True
2026-08-11 09:40:00+00:00,33.00%,52.00%,46.50%,19.00,-5.50,28.36%,-11.46%,True
2026-08-11 09:45:00+00:00,33.50%,52.00%,46.50%,18.50,-5.50,27.82%,-11.46%,True
2026-08-11 09:50:00+00:00,34.00%,52.00%,46.50%,18.00,-5.50,27.27%,-11.46%,True
2026-08-11 09:55:00+00:00,34.00%,51.50%,46.50%,17.50,-5.00,26.52%,-10.31%,True
2026-08-11 10:00:00+00:00,29.00%,51.00%,46.50%,22.00,-4.50,30.99%,-9.18%,True
